# Pipeline de Produção — Selagem FB14

Notebook orquestrador agendável como Seeq job.  
Executa a cada 1 hora: extrai dados, atualiza datasets analíticos, avalia gatilhos e grava no SharePoint.

**Fluxo**
- **Seção A — Dados Base**: `gen_sku_dates` → `gen_hour_prev` → `gen_sinais` → `02_sinais_forca.csv`
- **Seção B — Anomaly Delay**: puxa sinal de delay do Seeq, computa `stopped_h` → `anomaly_delay.csv`
- **Seção C — Weibull Operacional**: `gen_vida_rul(delay_csv)` → Weibull calibrado em horas operacionais → `gen_padroes`
- **Avaliação de gatilhos**: estado do ciclo → trigger engine → SharePoint

In [1]:
import sys
from pathlib import Path

# notebooks/ para mock seeq em dev local; lplcc_selagem/ para imports src.*
_NB_DIR = Path.cwd()
_ROOT   = _NB_DIR.parent
for _p in (_NB_DIR, _ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from seeq import spy
spy.jobs.schedule("every 1 hour")

,Schedule,Scheduled,Next Run
0,every 1 hour,Every hour,2026-06-05 18:00:00 BRT


,Schedule,Scheduled,Next Run
0,every 1 hour,Every hour,2026-06-05 18:00:00 BRT


In [ ]:
# ── Configuração ──────────────────────────────────────────────────────────────
MAQUINA     = "FB17"
LIST_NAME   = "Gatilhos_Selagem"
SP_URL      = ""    # defina SP_URL=https://... em sharepoint.ev
ESCREVER_SP = True  # alterar para True em produção

PATH_HOUR_PREV = _NB_DIR / "00_hour_prev.csv"
PATH_VIDA_RUL  = _NB_DIR / "01_vida_rul.csv"
PATH_WEIBULL   = _NB_DIR / "01_weibull_params.json"
PATH_SINAIS    = _NB_DIR / "02_sinais_forca.csv"
PATH_PADROES   = _NB_DIR / "03_padroes.csv"
PATH_SKU_DATES = _NB_DIR / "sku_dates.csv"
PATH_TROCA     = _NB_DIR / "troca_modulo.csv"
PATH_STATE     = _NB_DIR / "state_fb17.json"
PATH_DELAY     = _NB_DIR / "anomaly_delay.csv"

In [3]:
# ── Sync troca_modulo.csv ← SharePoint (CONSUMO MAINTACKER BCM.xlsx) ─────────
from src.sp_troca_sync import sync_troca_modulo

try:
    sync_troca_modulo(config_path=_ROOT / "config.yaml", out_path=PATH_TROCA)
except Exception as _e_sync:
    print(f"[sp_troca_sync] AVISO: falha ao sincronizar com SP — usando CSV local. ({_e_sync})")


[sp_troca_sync] Máquina: FB14
[sp_troca_sync] Baixando /Sites/H945/Suzano/api_csv/projeto_kairos/CONSUMO MAINTACKER BCM.xlsx ...
[sp_troca_sync] 13 trocas gravadas → /home/datalab/fb14-datalab/notebooks/troca_modulo.csv
               2025-06-11  indefinido
               2025-07-24  indefinido
               2025-08-03  indefinido
               2025-08-13  indefinido
               2025-09-22  indefinido
               2025-12-18  indefinido
               2026-02-02  indefinido
               2026-03-02  indefinido
               2026-03-22  indefinido
               2026-04-13  indefinido
               2026-05-06  indefinido
               2026-05-20  indefinido
               2026-05-29  indefinido


## Seção A — Dados base

Produz os artefatos que a Seção B (Anomaly Delay) vai precisar:
`sku_dates.csv`, `00_hour_prev.csv`, `02_sinais_forca.csv`.

In [ ]:
from src.generators.gen_sku_dates   import run as gen_sku_dates
from src.generators.gen_hour_prev   import run as gen_hour_prev
from src.generators.gen_sinais      import run as gen_sinais

print('[A.1] Atualizando Phantom dates via Seeq...')
df_sku_raw = gen_sku_dates(output_path=PATH_SKU_DATES)
n_phantom = df_sku_raw['phantom'].notna().sum()
print(f'      sku_dates: {len(df_sku_raw):,} leituras | {n_phantom:,} com Phantom Code')

print('[A.2] Extraindo dados do Seeq...')
df_hour = gen_hour_prev(output_path=PATH_HOUR_PREV, troca_csv=PATH_TROCA)
print(f'      00_hour_prev: {len(df_hour):,} leituras')

print('[A.3] Calculando features de força (Weibull de config.yaml — atualizado na Seção C)...')
df_sinais = gen_sinais(
    input_path=PATH_HOUR_PREV,
    output_path=PATH_SINAIS,
    troca_csv=PATH_TROCA,
)
print(f'      02_sinais_forca: {len(df_sinais):,} linhas')
print(f'      ciclos: {df_sinais["ciclo_id"].nunique()}  |  '
      f'periodo: {df_sinais["Timestamp"].min().date()} -> {df_sinais["Timestamp"].max().date()}')

## Seção B — Anomaly Delay

Puxa o sinal de Tempo Delay Calculado do Seeq e computa `stopped_h` acumulado por ciclo. Usa `02_sinais_forca.csv` (gerado na Seção A) para contextualizar os ciclos.

Resultado: `anomaly_delay.csv` com `stopped_h` por timestamp — insumo direto para o Weibull operacional na Seção C.

In [ ]:
from src.generators.gen_anomaly_delay import run as gen_anomaly_delay

print('[B.1] Puxando sinal de delay do Seeq e computando stopped_h por ciclo...')
df_anom = gen_anomaly_delay(
    output_path=PATH_DELAY,
    troca_csv=PATH_TROCA,
    config_path=_ROOT / 'config.yaml',
)
if df_anom.empty:
    print('      ⚠️  anomaly_delay vazio — Seção C usará horas de calendário')
else:
    print(f'      anomaly_delay: {len(df_anom):,} leituras | '
          f'stopped_h_max={df_anom["stopped_h"].max():.1f}h')

## Seção C — Weibull Operacional

Calibra o Weibull sobre horas **operacionais** (descontando paradas do `anomaly_delay.csv`). Produz `01_vida_rul.csv` e `01_weibull_params.json`.

Os parâmetros resultantes devem ser copiados para `config.yaml` (o `04_pipeline_mensageria.ipynb` alerta quando há divergência).

In [ ]:
from src.generators.gen_vida_rul  import run as gen_vida_rul
from src.generators.gen_padroes   import run as gen_padroes

print('[C.1] Calibrando Weibull em tempo operacional...')
df_vida = gen_vida_rul(
    input_path=PATH_HOUR_PREV,
    output_csv=PATH_VIDA_RUL,
    output_json=PATH_WEIBULL,
    troca_csv=PATH_TROCA,
    delay_csv=PATH_DELAY if PATH_DELAY.exists() else None,
)
n_ciclos = df_vida['ciclo_id'].nunique() if 'ciclo_id' in df_vida.columns else '?'
print(f'      01_vida_rul: {len(df_vida):,} linhas | {n_ciclos} ciclos')

import json as _json
_wp = _json.load(open(PATH_WEIBULL))
print(f'      Weibull: beta={_wp["weibull_beta"]:.4f}  eta_h={_wp["weibull_eta_h"]:.1f}h '
      f'({"operacional" if _wp.get("calibrado_em_horas_operacionais") else "calendário"})')

print('[C.2] Calculando padrões de detecção...')
df_padroes = gen_padroes(
    input_vida_rul=PATH_VIDA_RUL,
    input_sinais=PATH_SINAIS,
    output_path=PATH_PADROES,
    troca_csv=PATH_TROCA,
)
n_gen = df_padroes['genuino'].sum() if 'genuino' in df_padroes.columns else '?'
print(f'      03_padroes: {len(df_padroes)} ciclos ({n_gen} genuínos)')

In [ ]:
import pandas as pd

# 02_sinais_forca.csv já tem Media_norm gerada por gen_sinais (phantom auto-calibrado)
df_hourly = pd.read_csv(PATH_SINAIS, parse_dates=["Timestamp"])
df_hourly["Timestamp"] = pd.to_datetime(df_hourly["Timestamp"], utc=True)
df_hourly = df_hourly.set_index("Timestamp").sort_index()

n_norm    = df_hourly["Media_norm"].notna().sum() if "Media_norm" in df_hourly.columns else 0
n_total   = len(df_hourly)
cobertura = n_norm / n_total if n_total > 0 else 0.0
print(f"Media_norm: {n_norm:,}/{n_total:,} leituras ({cobertura:.0%})")

if n_norm > 0 and "phantom_fator" in df_hourly.columns:
    fator_medio  = df_hourly["phantom_fator"].mean()
    moda_phantom = df_hourly["phantom_codigo"].mode()
    phantom_dom  = moda_phantom.iloc[0] if not moda_phantom.empty else None
    print(f"Fator phantom médio : {fator_medio:.3f}")
    if phantom_dom is not None:
        print(f"Phantom dominante   : {phantom_dom}")

# Usa Media_norm se cobertura >= 50%; caso contrário, fallback para força bruta
media_col = "Media_norm" if cobertura >= 0.50 else "Media"
print(f"Coluna p/ trigger: '{media_col}'")

In [ ]:
# ── Contexto de paradas (anomaly_delay) ───────────────────────────────────
# PATH_DELAY já definido em Cell de configuração; gerado pela Seção B acima.
df_delay = None
if PATH_DELAY.exists():
    df_delay = pd.read_csv(PATH_DELAY, index_col="Timestamp", parse_dates=True)
    df_delay.index = df_delay.index.tz_convert("UTC")
    print(f"anomaly_delay: {len(df_delay):,} leituras | stopped_h_max={df_delay['stopped_h'].max():.1f}h")
else:
    print("anomaly_delay.csv não encontrado — age_risk sem correção de paradas")

## Etapa 3 — Estado do ciclo atual

In [ ]:
from src.predictor import load_troca_dates

troca_dates  = load_troca_dates(PATH_TROCA)
ultima_troca = troca_dates[-1]
hoje         = df_hourly.index[-1].normalize()
idade_dias   = (hoje - ultima_troca).days

print(f"Última troca confirmada : {ultima_troca.date()}")
print(f"Última leitura          : {hoje.date()}")
print(f"Idade do rolo           : {idade_dias} dias")

## Etapa 4 — Avaliação de gatilhos

In [ ]:
import os

sp_client = None
if ESCREVER_SP:
    try:
        from dotenv import dotenv_values
        # sharepoint.ev fica na raiz do projeto (fb14new/), não em notebooks/
        _ev = dotenv_values(_ROOT / "sharepoint.ev")
        _sp_url  = _ev.get("SP_URL") or "https://kimberlyclark.sharepoint.com/Sites/H945"
        _sp_user = _ev.get("SP_USER")
        _sp_pass = _ev.get("SP_PASS")
        from src.sharepoint_methods import SharePointClient
        sp_client = SharePointClient(url=_sp_url, username=_sp_user, password=_sp_pass)
        print(f"SharePoint: conectado ({_sp_url})")
    except Exception as _e:
        print(f"SharePoint: falha na conexão ({_e}) — modo dry-run")
else:
    print("SharePoint: dry-run (ESCREVER_SP=False)")

In [ ]:
from src.trigger_engine import TriggerEngine

engine = TriggerEngine(maquina=MAQUINA, state_path=PATH_STATE)

print(f"Avaliando {hoje.date()} | rolo com {idade_dias} dias | sinal='{media_col}'")
events = engine.evaluate(
    df_hourly=df_hourly,
    troca_date=ultima_troca,
    sp_client=sp_client,
    list_name=LIST_NAME,
    media_col=media_col,
    troca_dates=troca_dates,
    df_delay=df_delay,
)

print()
if events:
    print(f"{len(events)} disparo(s):")
    for ev in events:
        sp_tag = f" [SP ID={ev.sp_item_id}]" if ev.sp_item_id else ""
        print(f"  [{ev.gatilho}] {ev.severidade}{sp_tag}")
        print(f"  {ev.mensagem.splitlines()[0]}")
else:
    print("Nenhum gatilho disparado.")

In [ ]:
import json
# ── Kairos Snapshot + Efêmérides Kepler ───────────────────────────
# Envia estado atual ao SharePoint List Kairos_Snapshots (Kairos central)
import math

def _nivel_from_snap(snap, cfg):
    p = snap.get("p_risk", 0)
    if p >= cfg["trigger"]["limiar_p_risk"]: return "CRITICO"
    if p >= cfg["trigger"]["aviso_p_risk"]:  return "RISCO"
    return "NOMINAL"

def _severidade_from_snap(snap, cfg):
    p = snap.get("p_risk", 0)
    if p >= cfg["trigger"]["limiar_p_risk"]: return "CRITICA"
    if p >= cfg["trigger"]["aviso_p_risk"]:  return "ALTA"
    return "NOMINAL"

def _janela_label(p_risk):
    if p_risk >= 0.70: return "CRITICA"
    if p_risk >= 0.40: return "PLANEJADA"
    return "MONITORAR"

def _acao_recomendada(snap, cfg):
    p    = snap.get("p_risk", 0)
    dias = max(0.0, (cfg["trigger"]["weibull_eta_h"] / 24) - snap.get("age_days", 0))
    if p >= cfg["trigger"]["limiar_p_risk"]:
        return f"Substituicao urgente - p_risk={p:.2f}, dias restantes={dias:.0f}"
    if p >= cfg["trigger"]["aviso_p_risk"]:
        return f"Planejar substituicao - p_risk={p:.2f}, dias restantes={dias:.0f}"
    return f"Monitorar - p_risk={p:.2f}, dias restantes={dias:.0f}"

if ESCREVER_SP:
    try:
        from src.connector import load_config
        from src.trigger_engine import compute_p_risk_snapshot
        cfg = load_config(_ROOT / "config.yaml")

        snap_k   = compute_p_risk_snapshot(df_hourly, ultima_troca, hoje)
        eta_dias = cfg["trigger"]["weibull_eta_h"] / 24

        snapshot_dict = {
            "fonte":             "fb14-datalab",
            "maquina":           MAQUINA,
            "parte_mecanica":    "maintacker",
            "timestamp_iso":     snap_k.get("today", pd.Timestamp.now()).isoformat(),
            "idade_dias":        float(snap_k.get("age_days", 0)),
            "eta_ajustado_dias": eta_dias,
            "dias_restantes":    max(0.0, eta_dias - float(snap_k.get("age_days", 0))),
            "p_risk":            float(snap_k.get("p_risk", 0)),
            "age_risk":          float(snap_k.get("age_risk", 0)),
            "signal_score":      float(snap_k.get("sig_score", snap_k.get("signal_score", 0))),
            "nivel":             _nivel_from_snap(snap_k, cfg),
            "severidade":        _severidade_from_snap(snap_k, cfg),
            "mean_3d":           snap_k.get("mean_3d"),
            "slope_7d":          snap_k.get("slope_7d"),
            "proj_48h":          snap_k.get("proj_48h"),
            "urgencia_dias":     max(0.0, eta_dias - float(snap_k.get("age_days", 0))),
            "janela_label":      _janela_label(float(snap_k.get("p_risk", 0))),
            "acao_recomendada":  _acao_recomendada(snap_k, cfg),
        }

        # ── Efêmérides Kepler (M, E, e) ───────────────────────────────────────────
        # M = age_risk x 2pi  (Anomalia Media: previsao Weibull)
        # E = p_risk   x 2pi  (Anomalia Excentrica: risco hibrido observado)
        # e = (E - M) / sin(E)  [equacao de Kepler invertida: M = E - e*sin(E)]
        _M_rad = float(snap_k.get("age_risk", 0)) * 2 * math.pi
        _E_rad = float(snap_k.get("p_risk",   0)) * 2 * math.pi
        _sin_E = math.sin(_E_rad)
        _e_kep = min((_E_rad - _M_rad) / _sin_E, 0.99) if abs(_sin_E) > 1e-6 else 0.0
        _e_kep = max(_e_kep, 0.0)

        _sinal_atual = float(snap_k.get("mean_3d") or 0)
        _vel_sinal   = float(snap_k.get("slope_7d") or 0)  # N/dia
        _limite_inf  = float(
            cfg["trigger"].get("limiar_forca",
            cfg["trigger"].get("risco_min_forca", 800))
        )
        _dias_ate = (
            round((_sinal_atual - _limite_inf) / abs(_vel_sinal), 1)
            if _vel_sinal < 0 and _sinal_atual > _limite_inf else None
        )

        _coordenadas = {
            "anomalia_media":      round(float(snap_k.get("age_risk", 0)), 6),
            "eccentricidade":      round(_e_kep, 6),
            "anomalia_excentrica": round(float(snap_k.get("p_risk", 0)), 6),
            "velocidade_sinal":    round(_vel_sinal, 4),
            "sinal_atual":         round(_sinal_atual, 2),
            "limite_inferior":     _limite_inf,
            "dias_ate_limite":     _dias_ate,
            "unidade_sinal":       "N",
            "eta_dias":            eta_dias,
            "beta":                float(cfg["trigger"].get("weibull_beta", 1.181)),
        }
        snapshot_dict["coordenadas"] = json.dumps(_coordenadas, ensure_ascii=False)

        _ids    = sp_client.insert_list_item("Kairos_Snapshots", [snapshot_dict])
        item_id = _ids[0] if _ids else None
        print(f"[kairos] Snapshot -> Kairos_Snapshots ID={item_id}")
    except Exception as _e_kairos:
        print(f"[kairos] Aviso: snapshot nao enviado - {_e_kairos}")
else:
    print("[kairos] ESCREVER_SP=False - snapshot nao enviado (dry-run)")


## Diagnóstico (execução manual)

Snapshot completo dos indicadores calculados para hoje. Útil para debug e auditoria.

In [ ]:
from src.trigger_engine import compute_p_risk_snapshot

snap = compute_p_risk_snapshot(df_hourly, ultima_troca, hoje)
print("Indicadores calculados:")
for k, v in snap.items():
    print(f"  {k:<22}: {v}")

In [ ]:
# ── Histórico de gatilhos por ciclo ────────────────────────────────────────
#   0 = ciclo atual (aberto)
#  -1 = último ciclo completo
#  -2 = penúltimo  ...
CICLO_OFFSET = -1

import tempfile, json
from pathlib import Path
from src.trigger_engine import TriggerEngine, compute_p_risk_snapshot

# ── Resolver limites do ciclo ─────────────────────────────────────────────
_n = len(troca_dates)
_min_offset = -(_n - 1)   # ex: 15 trocas → offset mínimo = -14

if CICLO_OFFSET > 0 or CICLO_OFFSET < _min_offset:
    raise ValueError(
        f"CICLO_OFFSET deve estar entre {_min_offset} e 0 "
        f"(há {_n - 1} ciclos completos + ciclo atual)."
    )

if CICLO_OFFSET == 0:
    t_ini  = troca_dates[-1]
    t_fim  = None
    rotulo = f"atual (desde {t_ini.date()})"
else:
    t_ini  = troca_dates[CICLO_OFFSET - 1]
    t_fim  = troca_dates[CICLO_OFFSET]
    _dur   = (t_fim - t_ini).days
    rotulo = f"ciclo {CICLO_OFFSET:+d}  |  {t_ini.date()} → {t_fim.date()}  ({_dur} dias)"

print(f"━━ Ciclo selecionado: {rotulo} ━━\n")

# ── Filtrar séries do ciclo ───────────────────────────────────────────────
_df_c = df_hourly[df_hourly.index >= t_ini]
if t_fim:
    _df_c = _df_c[_df_c.index < t_fim]

if _df_c.empty:
    print("  Sem dados para este ciclo.")
else:
    # Mini-backtest: reconstruir eventos disparados no ciclo
    _state_tmp = Path(tempfile.mktemp(suffix=".json"))
    _eng       = TriggerEngine(MAQUINA, _state_tmp)
    _dias      = pd.DatetimeIndex(sorted({ts.normalize() for ts in _df_c.index}))

    _eventos = []   # lista de (day, TriggerEvent)
    for _day in _dias:
        _df_ate = df_hourly[df_hourly.index <= _day + pd.Timedelta(hours=23, minutes=59)]
        try:
            for _ev in _eng.evaluate(
                _df_ate,
                troca_date=t_ini,
                sp_client=None,
                today=_day,
                troca_dates=troca_dates,
                media_col=media_col,
                df_delay=df_delay,
            ):
                _eventos.append((_day, _ev))
        except Exception as _exc:
            print(f"  [ERRO] {_day.date()}: {_exc}")

    if _state_tmp.exists():
        _state_tmp.unlink()

    # ── Sumário ─────────────────────────────────────────────────────────
    if not _eventos:
        print("  Nenhum gatilho disparado neste ciclo.")
    else:
        print(f"  {len(_eventos)} disparo(s):\n")
        for _d, _ev in _eventos:
            _mark = "  ◀ último" if (_d, _ev) is _eventos[-1] else ""
            print(f"    {_d.date()}  [{_ev.severidade:<6}]  {_ev.gatilho}{_mark}")

        # ── Detalhe do último disparo ─────────────────────────────────
        _day_ult, _ev_ult = _eventos[-1]
        print(f"\n── Último disparo: {_ev_ult.gatilho}  "
              f"em {_day_ult.date()}  (dia {_ev_ult.idade_maintacker} do ciclo) ──\n")

        _snap = compute_p_risk_snapshot(df_hourly, t_ini, _day_ult)
        for _k, _v in _snap.items():
            print(f"  {_k:<22}: {_v}")

        print("\n── Payload enviado (TeamsPayload) ──")
        if _ev_ult.teams_payload:
            try:
                print(json.dumps(json.loads(_ev_ult.teams_payload), indent=2, ensure_ascii=False))
            except Exception:
                print(_ev_ult.teams_payload)
        else:
            print("  (sem payload — dry-run)")